# Atividade extra · Regressão logística
## Um grão, duas decisões: é HOROZ ou outra variedade?

**Valor: até 1,0 ponto extra · Individual ou em equipes de até 5 pessoas**

**Prazo:** a ser informado pelo professor.  
**Entrega:** um notebook por pessoa/equipe, com código, saídas, gráficos e respostas.  
**Tempo estimado:** 3 a 4 horas, conforme a familiaridade com as ferramentas.

Você investigará uma base real, construirá um alvo desbalanceado e usará
**regressão logística para classificação binária**. Complete os blocos `TODO`
e escreva suas interpretações nas células Markdown. As verificações do roteiro
só passam depois que os respectivos blocos forem preenchidos.


### Identificação

**Turma:** preencher  
**Nome da equipe, se houver:** preencher

| Integrante | Matrícula | Contribuição na atividade |
|---|---|---|
| Preencher | Preencher | Preencher |

Acrescente linhas até o limite de cinco integrantes. Em trabalho individual,
preencha apenas uma linha. Todos os integrantes devem compreender o fluxo
completo e as conclusões apresentadas.


### O problema

Imagine uma linha de separação de grãos que recebe medidas extraídas de
imagens. A equipe quer identificar a variedade **HOROZ** e encaminhá-la para
um lote específico. Este é um cenário didático baseado em dados reais.

A base **Dry Bean** contém **13.611 grãos**, **16 medidas numéricas** e rótulos
de **sete variedades**. Trabalharemos com:

- **Classe 1 — positivo:** HOROZ.
- **Classe 0 — negativo:** qualquer uma das outras seis variedades.

A proporção entre essas classes será calculada por você. Não reduza a base a
apenas duas variedades: use todos os registros válidos e reúna as outras seis
na classe 0. O CSV preserva o rótulo multiclasse `Class` para uma atividade
futura; **nesta entrega, o alvo deve conter somente 0 e 1**.

Ao final, você deverá explicar se o modelo encontra a classe minoritária,
quais erros comete e o que a acurácia deixa de mostrar.


### Ambiente e arquivos

**Jupyter:** mantenha o notebook e a pasta `dados/` juntos. O carregador também
funciona quando o diretório atual é a raiz do projeto. Para instalar as
bibliotecas, use `python -m pip install -r requirements.txt` no terminal,
dentro da pasta `atividade/`.

**Google Colab:** abra este notebook e envie `dry_bean.csv` e
`dicionario_dry_bean.csv` para a área de arquivos da sessão. As bibliotecas
necessárias normalmente já estão disponíveis no Colab.

O carregamento abaixo está pronto. **O diagnóstico, a EDA e os modelos devem
ser implementados por você.** Acrescente células quando necessário.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
ATRIBUTOS = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRation", "Eccentricity", "ConvexArea", "EquivDiameter",
    "Extent", "Solidity", "roundness", "Compactness",
    "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4",
]

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": False})
print(f"pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")


In [ ]:
def localizar_arquivo(nome):
    candidatos = [
        Path("dados") / nome,
        Path(nome),
        Path("atividade/dados") / nome,
    ]
    caminho = next((p for p in candidatos if p.is_file()), None)
    if caminho is None:
        raise FileNotFoundError(
            f"Arquivo {nome} não encontrado. Mantenha-o em dados/ "
            "ou envie-o para a pasta atual no Colab."
        )
    return caminho


df_bruto = pd.read_csv(localizar_arquivo("dry_bean.csv"))
dicionario = pd.read_csv(localizar_arquivo("dicionario_dry_bean.csv"))

assert list(df_bruto.columns) == ATRIBUTOS + ["Class"], "Confira o CSV utilizado."
print("Base original carregada. Preserve df_bruto para registrar suas decisões.")
display(df_bruto.head())
display(dicionario)


**Fonte e licença:** [Dry Bean — UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset),
[DOI 10.24432/C50S4B](https://doi.org/10.24432/C50S4B), licença
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
Estudo de Murat Koklu e Ilker Ali Ozkan (2020),
[DOI 10.1016/j.compag.2020.105507](https://doi.org/10.1016/j.compag.2020.105507).
A cópia local foi obtida em 05/11/2026 e convertida de ARFF para CSV, mantendo
as linhas, os valores e os nomes originais. Consulte `dados/FONTE.md`.

**Atenção aos nomes:** o arquivo usa `AspectRation` e `roundness` exatamente
com essas grafias. Não são colunas novas nem erros do carregador.


## 1. Contexto, diagnóstico e alvo · 0,10 ponto

**1.1 Diagnóstico da base original.** Apresente:

- dimensões, tipos das colunas e resumo de valores ausentes;
- quantidade de linhas integralmente duplicadas;
- contagem das sete variedades em `Class`.

Nesta etapa, use a base completa apenas para verificar sua integridade e a
composição dos rótulos. A EDA detalhada das medidas virá depois da separação.


In [ ]:
# TODO 1A: use df_bruto para apresentar o diagnóstico solicitado.
# Sugestões: .shape, .info(), .isna().sum(), .duplicated().sum(), .value_counts().


**1.2 Preparação estrutural.** Crie `df` a partir de `df_bruto`, removendo
**somente linhas integralmente duplicadas** e reiniciando o índice. Registre
quantas linhas foram retiradas. Esta é uma regra fixa do exercício para que
cópias exatas não apareçam no treino e no teste.

Não remova grãos só por terem medidas extremas. Se encontrar ausências ou
valores impossíveis, descreva a situação antes de propor um tratamento.
Qualquer transformação que aprenda parâmetros deverá ser ajustada no treino.

**1.3 Alvo e entradas.** Crie:

- `y`: série de inteiros, com 1 para `Class == "HOROZ"` e 0 para as demais;
- `X`: DataFrame com **somente as 16 colunas de `ATRIBUTOS`**.

`Class` revela a resposta. O alvo binário, o índice e qualquer coluna derivada
do rótulo também não podem entrar em `X`.


In [ ]:
# TODO 1B: substitua None pelas expressões de preparação, entradas e alvo.
df = None
X = None
y = None

assert isinstance(df, pd.DataFrame), "Crie df após a retirada de duplicatas exatas."
assert df.duplicated().sum() == 0, "Ainda existem linhas integralmente duplicadas."
assert len(df) == len(df_bruto.drop_duplicates()), "Confira a regra de preparação."
assert df.index.equals(pd.RangeIndex(len(df))), "Reinicie o índice após a preparação."
assert isinstance(X, pd.DataFrame) and list(X.columns) == ATRIBUTOS
assert isinstance(y, pd.Series) and set(y.unique()) == {0, 1}
assert pd.api.types.is_integer_dtype(y), "Converta o alvo para inteiros."
assert X.index.equals(y.index) and X.index.equals(df.index)
assert X.equals(df[ATRIBUTOS]), "Preserve as medidas originais nas entradas."
assert np.array_equal(y.to_numpy(), df["Class"].eq("HOROZ").astype(int).to_numpy())
print("Entradas e alvo conferidos. Registre o número de linhas removidas.")


**Responda:** o que cada linha representa? Por que esta tarefa é de
classificação, mesmo usando um método chamado regressão logística?
Explique o vazamento de informação que ocorreria ao incluir `Class` em `X`
e justifique a retirada de duplicatas antes da divisão.

**Sua resposta:** preencher.


## 2. Separe os dados antes da EDA detalhada · 0,10 ponto

Use `train_test_split` com **20% para teste**, `random_state=RANDOM_STATE` e
`stratify=y`. Guarde os resultados em `X_treino`, `X_teste`, `y_treino` e
`y_teste`. Apresente uma tabela com o tamanho e a proporção da classe 1 em
cada parte.

**Reserve o teste até a etapa 6.** Conferir seu tamanho e a proporção dos
rótulos nesta etapa é permitido. Não use suas medidas, erros ou métricas
para escolher variáveis, pesos de classe ou limite de decisão.


In [ ]:
# TODO 2: faça a separação estratificada e mostre a tabela solicitada.
X_treino, X_teste, y_treino, y_teste = None, None, None, None

assert all(isinstance(z, pd.DataFrame) for z in [X_treino, X_teste]), "Faça a divisão."
assert all(isinstance(z, pd.Series) for z in [y_treino, y_teste])
assert len(X_treino) + len(X_teste) == len(X)
assert len(X_teste) == int(np.ceil(0.20 * len(X)))
assert X_treino.index.intersection(X_teste.index).empty
assert X_treino.index.equals(y_treino.index) and X_teste.index.equals(y_teste.index)
assert set(y_treino.unique()) == set(y_teste.unique()) == {0, 1}
assert abs(y_treino.mean() - y_teste.mean()) < 0.01


**Responda:** por que a estratificação é útil em uma base desbalanceada?
O que poderia acontecer se você separasse os dados pela ordem das linhas,
sem embaralhar? Por que o teste precisa ficar reservado?

**Sua resposta:** preencher.


## 3. Análise exploratória de dados — EDA · 0,25 ponto

Trabalhe **somente com `X_treino` e `y_treino`** nesta etapa. Use o dicionário
para explicar as medidas escolhidas. O objeto abaixo é apenas um apoio para
gráficos e agrupamentos; não o utilize como entrada do modelo.

**3.1 Desbalanceamento:** apresente contagens e porcentagens das classes,
um gráfico de barras e a razão entre a quantidade da classe majoritária e
a da minoritária. Interprete o tamanho do desbalanceamento.

**3.2 Distribuições:** apresente estatísticas descritivas e histogramas de
pelo menos **duas medidas**, incluindo uma de tamanho e uma de formato.
Investigue diferenças de escala, assimetrias e valores extremos.

**3.3 Comparação entre classes:** faça boxplots de pelo menos **duas medidas**
por classe e compare as medianas desses atributos. Discuta a separação e a
sobreposição entre os grupos.

**3.4 Relações entre medidas:** apresente uma matriz de correlação numérica
com mapa de calor para **pelo menos seis atributos**. Aponte um par que
expresse informação semelhante e explique por que isso afeta a interpretação
isolada dos coeficientes. Correlação não estabelece causalidade.

Produza pelo menos **quatro figuras**: classes, histogramas, boxplots e mapa
de calor. Cada figura deve ter título, identificação dos eixos e legenda
quando necessária. Escreva uma interpretação junto de cada resultado.


In [ ]:
eda_treino = X_treino.copy()
eda_treino["eh_horoz"] = y_treino
nomes_classes = {0: "Outras variedades", 1: "HOROZ"}
# Este DataFrame inclui o alvo e serve apenas para a EDA.


In [ ]:
# TODO 3A: contagens, porcentagens, razão majoritária/minoritária e gráfico de barras.
# Use apenas os rótulos de y_treino.


**Interpretação do desbalanceamento:** preencher com os números encontrados.


In [ ]:
# TODO 3B: estatísticas e histogramas de uma medida de tamanho e outra de formato.
# Sugestões: Area ou MajorAxisLength; AspectRation, roundness ou Compactness.


**Interpretação das distribuições e escalas:** preencher.
Um valor extremo é necessariamente um erro de coleta? Justifique.


In [ ]:
# TODO 3C: boxplots e medianas de pelo menos duas medidas, separados por eh_horoz.


**Interpretação da comparação entre classes:** preencher.
Quais medidas parecem ajudar a identificar HOROZ? Há sobreposição?


In [ ]:
# TODO 3D: matriz de correlação e mapa de calor de pelo menos seis atributos.
# Use apenas as medidas de X_treino; exclua o alvo dessa matriz.
# Dica de visualização: ax.imshow(matriz, vmin=-1, vmax=1, cmap="coolwarm").


**Interpretação das correlações:** preencher.

**Síntese da EDA:** escreva três achados, cada um com uma evidência numérica,
sua interpretação e uma limitação. Relacione ao menos um achado ao preparo
do modelo. Use os 16 atributos no experimento obrigatório; não é necessário
implementar seleção de variáveis.

1. Preencher.
2. Preencher.
3. Preencher.


## 4. Construa uma referência majoritária · 0,10 ponto

Antes da regressão logística, crie `referencia` usando
`DummyClassifier(strategy="most_frequent")` e ajuste com o treino.
Esse classificador sempre prevê a classe mais frequente que viu no ajuste.

Mostre qual classe ele aprendeu a prever. Calcule **no treino**, apenas como
diagnóstico, acurácia e recall da classe 1. Não interprete esses valores como
estimativa de generalização. Na etapa 6, a referência será avaliada no teste.

Para métricas de precisão, recall e F1 da classe positiva, use `pos_label=1`.
Quando não houver previsões positivas, utilize `zero_division=0` e explique
esse caso ao interpretar os resultados.


In [ ]:
# TODO 4: crie e ajuste a referência. Mostre a classe prevista e as métricas no treino.
referencia = None

assert isinstance(referencia, DummyClassifier), "Crie o classificador de referência."
assert referencia.strategy == "most_frequent"
assert hasattr(referencia, "classes_"), "Ajuste a referência somente no treino."


**Responda:** por que um modelo que sempre prevê a classe majoritária pode
ter acurácia aparentemente alta? Ele atende ao objetivo de identificar
HOROZ? Relacione a resposta ao recall encontrado.

**Sua resposta:** preencher.


## 5. Regressão logística e validação · 0,20 ponto

**5.1 Dois pipelines.** Crie o dicionário `modelos` com duas opções:

- `sem_pesos`: `StandardScaler` seguido de `LogisticRegression`, com
  `max_iter=2000`, `C=1.0` e `class_weight=None`;
- `com_pesos`: o mesmo pipeline, alterando somente para
  `class_weight="balanced"`.

Use os nomes de etapa `"escala"` e `"logistica"`. Mantenha as mesmas entradas
e o limite **0,5** para ambas as opções. O ajuste de pesos modifica a
importância das classes no treinamento; não cria novos grãos e não garante
melhora em todas as métricas.

**5.2 Comparação sem abrir o teste.** Use `StratifiedKFold` com **5 partes**,
`shuffle=True` e `random_state=RANDOM_STATE`. Aplique `cross_validate` a cada
**pipeline completo**, exclusivamente em `X_treino, y_treino`, reutilizando
o mesmo particionamento. Apresente média e desvio padrão de **precisão,
recall e F1 da classe 1**. A acurácia pode aparecer como informação adicional.

**Critério fixo:** escolha a maior média de F1 da classe 1. Em caso de empate,
escolha `sem_pesos`. Esse critério foi definido antes da avaliação de teste.
Registre a escolha em `nome_escolhido` e justifique com a tabela.

A padronização deve ser aprendida novamente dentro de cada parte da validação.
Não aplique `fit_transform` à base inteira ou ao treino antes de chamar
`cross_validate`: passe o pipeline completo.


In [ ]:
# TODO 5A: crie os dois pipelines no dicionário modelos.
modelos = {}

assert set(modelos) == {"sem_pesos", "com_pesos"}, "Crie as duas opções solicitadas."
for nome, estimador in modelos.items():
    assert isinstance(estimador, Pipeline)
    assert list(estimador.named_steps) == ["escala", "logistica"]
    assert isinstance(estimador.named_steps["escala"], StandardScaler)
    assert isinstance(estimador.named_steps["logistica"], LogisticRegression)
    peso_esperado = None if nome == "sem_pesos" else "balanced"
    assert estimador.named_steps["logistica"].class_weight == peso_esperado
    assert estimador.named_steps["logistica"].max_iter == 2000
    assert estimador.named_steps["logistica"].C == 1.0


In [ ]:
# TODO 5B: configure a validação estratificada e compare os pipelines no treino.
# Dica: make_scorer(precision_score, pos_label=1, zero_division=0)
# permite definir como tratar a ausência de previsões positivas.
# Guarde uma tabela com médias e desvios de precisão, recall e F1.
# Registre nome_escolhido usando o critério de maior média de F1.
nome_escolhido = None

assert nome_escolhido in modelos, "Escolha uma opção usando somente a validação no treino."


**Responda:**

1. Por que padronizar as medidas? Por que o `StandardScaler` precisa ficar
   dentro do pipeline durante a validação cruzada?
2. Qual foi o efeito de `class_weight="balanced"` sobre precisão, recall e F1?
   Apoie a resposta nos valores encontrados, mesmo se a opção com pesos
   não apresentar o maior F1.
3. Qual opção foi escolhida? O desvio padrão sugere estabilidade entre as
   partes? A média de validação garante o desempenho em dados futuros?

**Suas respostas:** preencher antes de executar a etapa 6.


**5.3 Ajuste final.** Crie `modelo_final` como um `clone` da opção escolhida e
ajuste com todo o treino. Os modelos ajustados em cada parte da validação
não substituem esse ajuste final. Preserve o teste para a próxima etapa.


In [ ]:
# TODO 5C: clone a opção escolhida e ajuste em X_treino, y_treino.
modelo_final = None

assert isinstance(modelo_final, Pipeline), "Crie o pipeline final."
assert hasattr(modelo_final.named_steps["logistica"], "classes_"), "Faça o ajuste final."
assert np.array_equal(modelo_final.classes_, [0, 1])
assert np.allclose(
    modelo_final.named_steps["escala"].mean_, X_treino.mean().to_numpy()
), "A escala final deve ser aprendida somente no conjunto de treino."


## 6. Avaliação final no teste · 0,15 ponto

Agora avalie **a opção já escolhida e a referência majoritária** em
`X_teste, y_teste`. Após observar esses resultados, não troque o modelo,
as variáveis ou o limite para melhorar os números da entrega.

**6.1 Probabilidades e classes.** Obtenha as classes previstas pelo modelo e
a probabilidade da classe 1. Confira a ordem das colunas em
`modelo_final.classes_`. Mostre uma tabela com **cinco exemplos**, incluindo
classe real, probabilidade de HOROZ e classe prevista. Uma probabilidade é
uma estimativa do modelo, não uma certeza sobre aquele grão.

**6.2 Comparação.** Apresente uma tabela para a referência e a regressão com:

- acurácia;
- precisão, recall e F1 da **classe 1**;
- acurácia balanceada, que corresponde à média dos recalls das duas classes.

**6.3 Erros.** Mostre a matriz de confusão do modelo final com a ordem
`labels=[0, 1]` e os eixos identificados como real e previsto. Apresente
também o `classification_report` com nomes de classe e `zero_division=0`.

O resultado numérico do modelo **não determina a nota**. Serão avaliados
o procedimento, a comparação com a referência e a qualidade da interpretação.


In [ ]:
# TODO 6A: obtenha previsões e P(classe 1), mantendo o limite fixo de 0,5.
# Depois, monte a tabela dos cinco exemplos de teste.
y_previsto = None
p_horoz = None

assert y_previsto is not None and p_horoz is not None, "Gere classes e probabilidades."
assert len(y_previsto) == len(p_horoz) == len(y_teste)
assert set(np.unique(y_previsto)).issubset({0, 1})
assert np.all((np.asarray(p_horoz) >= 0) & (np.asarray(p_horoz) <= 1))
assert np.array_equal(y_previsto, (np.asarray(p_horoz) >= 0.5).astype(int))


In [ ]:
# TODO 6B: preveja com a referência no MESMO teste e faça a tabela de métricas.
# Use pos_label=1 e zero_division=0 nas métricas da classe positiva.
# A acurácia balanceada é obtida com balanced_accuracy_score.


In [ ]:
# TODO 6C: matriz de confusão, visualização e classification_report do modelo final.
# Ordem dos rótulos: [0, 1]. Linhas = real; colunas = previsto.
# Identifique VN, FP, FN e VP e confira a soma com o tamanho do teste.


**Interprete os resultados:**

1. Quantos grãos HOROZ foram encontrados e quantos deixaram de ser encontrados?
   Indique os números da matriz de confusão usados na resposta.
2. Neste cenário, o que significam um falso positivo e um falso negativo?
   Se o objetivo fosse evitar que HOROZ fosse encaminhado para o lote das
   demais variedades, qual erro mereceria mais atenção e qual métrica o mede?
3. O modelo supera a referência no objetivo da atividade? Compare pelo menos
   duas métricas, incluindo o recall da classe 1. Explique por que olhar
   apenas para a acurácia seria insuficiente.
4. Escolha um dos cinco exemplos e explique a diferença entre sua
   probabilidade estimada e sua classe prevista.

**Suas respostas:** preencher.


## 7. Conclusão e entrega · 0,10 ponto

Escreva **de 8 a 12 linhas** que conectem o problema aos resultados:

- defina o alvo e informe a proporção da classe 1 no treino;
- retome dois achados da EDA e a importância da padronização;
- informe a opção escolhida e compare F1 e recall do modelo e da referência
  no teste;
- descreva o principal tipo de erro observado, uma limitação da análise e
  uma próxima investigação que poderia ajudar.

Considere que registros desta base não representam automaticamente grãos
de outros lotes, condições de imagem ou equipamentos. Discuta a generalização
com base no desenho do experimento, sem afirmar que o sistema está pronto
para uso na linha de separação.

**Sua conclusão:** preencher.


### Rubrica de correção — até 1,0 ponto extra

| Etapa | Evidência esperada | Pontos |
|---|---|---:|
| 1. Contexto, diagnóstico e alvo | Diagnóstico, registro da deduplicação, alvo correto e ausência de rótulos em X | 0,10 |
| 2. Separação | Teste de 20%, estratificação, reprodutibilidade e justificativa da reserva do teste | 0,10 |
| 3. EDA | Desbalanceamento (0,05), distribuições (0,05), comparação entre classes (0,05), correlações (0,05) e síntese interpretada (0,05) | 0,25 |
| 4. Referência | Implementação majoritária e discussão de acurácia e recall | 0,10 |
| 5. Regressão logística | Pipelines (0,05), validação e comparação com pesos (0,10), escolha justificada e ajuste final (0,05) | 0,20 |
| 6. Avaliação | Probabilidades, tabela comparativa, matriz de confusão e interpretação contextual dos erros | 0,15 |
| 7. Conclusão e organização | Conclusão sustentada por evidências, limitações, identificação e execução organizada | 0,10 |
| **Total** | | **1,00** |

Entregas parciais podem receber pontuação proporcional às evidências
apresentadas. Gráficos e números sem interpretação não atendem integralmente
aos critérios. A mesma rubrica vale para trabalho individual ou em equipe.


### Desafios opcionais — sem acréscimo além do limite de 1,0 ponto

Escolha um, se desejar aprofundar:

- **Limite de decisão:** use `cross_val_predict(..., method="predict_proba")`
  com o pipeline e as mesmas cinco partes **somente no treino** para estudar
  os limites 0,3, 0,5 e 0,7. Compare precisão e recall nessas previsões de
  validação. Não escolha o limite com o teste e não refaça a avaliação final
  obrigatória após examiná-lo.
- **Coeficientes:** apresente os coeficientes do modelo final e interprete o
  sinal de dois deles na escala padronizada. Considere a regularização e
  as correlações entre medidas; não interprete magnitude como prova de
  causalidade ou importância universal.
- **Curva precisão–recall:** construa-a com probabilidades de validação no
  treino e calcule `average_precision_score`. Explique como ela complementa
  a avaliação com o limite fixo. A AP não é a precisão em um único limite.

A classificação multiclasse fica para outra atividade. Não implemente um
modelo com sete classes nesta entrega.


In [ ]:
# OPCIONAL: implemente aqui um dos desafios, se desejar.


### Checklist antes de entregar

- [ ] Identifiquei todos os integrantes, com matrícula e contribuição; são no máximo cinco.
- [ ] Completei os blocos obrigatórios e todas as respostas em Markdown.
- [ ] Registrei ausências e duplicatas e criei o alvo 0/1 conforme o enunciado.
- [ ] Mantive apenas os 16 atributos em X; excluí Class e qualquer derivação do alvo.
- [ ] Separei o teste antes da EDA detalhada e usei somente o treino na exploração.
- [ ] Apresentei as quatro figuras exigidas e interpretei os resultados.
- [ ] Comparei os pipelines com validação cruzada no treino e apliquei o critério de F1.
- [ ] Avaliei a opção escolhida e a referência no mesmo teste, sem selecionar com ele.
- [ ] Interpretei a classe minoritária e os falsos positivos/negativos.
- [ ] Reiniciei o kernel e executei todas as células em ordem, sem erros.
- [ ] Salvei o notebook com tabelas e gráficos visíveis.

**Nome do arquivo:** `Atividade_Extra_Logistica_NomeOuEquipe.ipynb`.
Entregue esse notebook no canal indicado pelo professor. Se criar arquivos
auxiliares necessários à execução, envie-os junto e documente seu uso.
Não é necessário reenviar os CSVs originais fornecidos.


### Referências de consulta

- [Dry Bean — UCI e documentação da base](https://archive.ics.uci.edu/dataset/602/dry+bean+dataset).
- [Regressão logística e class_weight](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html).
- [Preparo dos dados e StandardScaler](https://scikit-learn.org/stable/modules/preprocessing.html#standardization-or-mean-removal-and-variance-scaling).
- [Validação cruzada](https://scikit-learn.org/stable/modules/cross_validation.html).
- [Métricas de classificação](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).

Use a documentação e os materiais da disciplina para apoiar a implementação.
Registre as fontes adicionais consultadas e explique as decisões adotadas.
